# Raw Excel files - Wiertsema

This notebook lists the raw Excel files provided by Wiertsema in `input_data/Wiertsema` and shows a quick preview of the first file (if any).

Run from the repository root or the notebook will attempt to detect the repo root automatically. Use the Poetry venv (e.g. `poetry shell`) so the correct Python environment and dependencies (pandas, openpyxl) are available.

In [1]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / '.git').exists() or (candidate / 'pyproject.toml').exists():
            return candidate
    return p

repo_root = find_repo_root()
in_dir = repo_root / 'input_data' / 'Wiertsema'
files = sorted(in_dir.glob('*.xlsx')) + sorted(in_dir.glob('*.xls'))
print('Repository root:', repo_root)
print('Looking in:', in_dir)
print(f'Found {len(files)} Excel file(s)')
for p in files[:20]:
    print('-', p.name)

Repository root: D:\Users\jvanruitenbeek\data_validation
Looking in: D:\Users\jvanruitenbeek\data_validation\input_data\Wiertsema
Found 11 Excel file(s)
- Beemster_86349_1_deel_1.xlsx
- Beemster_86349_1_deel_2.xlsx
- Beemster_oud_83034_1.xlsx
- Belmermeer_87097_1.xlsx
- gecombineerde_peilbuisdata_4_oude_raaien_beemster_26032026 1.xlsx
- peilbuisdata_alle_sensoren_07042026.xlsx
- Purmer_87074_1.xlsx
- Schellingwoude _84507_1.xlsx
- Schellingwoude_84507_2.xlsx
- Schermer_88111_1_deel_1.xlsx
- Schermer_88111_1_deel_2.xlsx


In [2]:
import os
from pathlib import Path
from openpyxl import load_workbook

# Use correct relative path from the notebooks folder
folder = Path('../input_data/Wiertsema').resolve()
if not folder.exists():
    raise FileNotFoundError(f"Map niet gevonden: {folder}")

files = [f for f in os.listdir(folder) if f.endswith('.xlsx')]
for fname in files:
    print(f'Bestand: {fname}')
    wb = load_workbook(folder / fname, data_only=True, read_only=True)
    for sheet in wb.sheetnames:
        print(f'  Tabblad: {sheet}')
        ws = wb[sheet]
        for i, row in enumerate(ws.iter_rows(values_only=True), 1):
            print(row)
            if i >= 5:
                break
        print()

Bestand: Beemster_86349_1_deel_1.xlsx
  Tabblad: 86349-1 MB049PB01 (B_BE0377+32_
('86349-1 MB049PB01 (B_BE0377+32_BIKR_GMW_PB1_F-252)', 'MSLV8_o54177', None, None)
('Timestamp', 'Water Absolute Druk (Pa)', 'Water Temperatuur (ºC)', 'Waterniveau (m NAP)')
(datetime.datetime(2024, 8, 14, 17, 0), 109860, None, -2.007)
(datetime.datetime(2024, 8, 14, 18, 0), 109880, None, -2.007)
(datetime.datetime(2024, 8, 14, 19, 0), 109850, None, -2.009)

  Tabblad: 86349-1 MB047PB01 (B_BE0328+2_B
('86349-1 MB047PB01 (B_BE0328+2_BIT_GMW_PB1_F-651)', 'MSLV8_o54151', None, None)
('Timestamp', 'Water Absolute Druk (Pa)', 'Water Temperatuur (ºC)', 'Waterniveau (m NAP)')
(datetime.datetime(2024, 8, 14, 16, 0), 123045, 10, -4.543)
(datetime.datetime(2024, 8, 14, 16, 38, 40, 597000), 122695, 10, -4.58)
(datetime.datetime(2024, 8, 14, 16, 39, 41, 113000), 122795, 10, -4.569)

  Tabblad: 86349-1 MB045PB01 (B_BE0328+3_B
('86349-1 MB045PB01 (B_BE0328+3_BUKR_GMW_PB1_F-349)', 'MSLV8_o54169', None, None)
('Timestamp'

In [3]:
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.utils.datetime import from_excel
import pandas as pd
import re
from datetime import datetime

# --------- Config ----------
in_folder = Path('../input_data/Wiertsema').resolve()
dataset_root = Path('../output_data/wiertsema').resolve()
dataset_root.mkdir(parents=True, exist_ok=True)
# ---------------------------

def sanitize_filename(name: str) -> str:
    s = (name or "").replace("(", "").replace(")", "")
    s = re.sub(r'[<>:"/\\|?*]+', "_", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip(" ._")
    return s or "series"

def norm_header(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = s.replace("(", "").replace(")", "")
    return re.sub(r"[^a-z0-9]+", "", s)

def first_non_empty(values):
    for v in values:
        if v not in (None, ""):
            return str(v)
    return "sheet"

def to_datetime(v, wb_epoch):
    if isinstance(v, datetime):
        return v
    if isinstance(v, (int, float)):
        try:
            return from_excel(v, wb_epoch)
        except Exception:
            return None
    return None

def detect_sensor_groups(title_row, header_row):
    """
    Detect sensor column groups from Row 1 (title) and Row 2 (headers).
    Non-None values in title_row[1:] mark the start of each sensor's columns.
    Returns list of {"sensor_id": str, "wn_idx": int|None, "start_col": int, "end_col": int}.
    """
    sensor_positions = []
    for col_idx in range(1, len(title_row)):
        val = title_row[col_idx]
        if val is not None and str(val).strip() != "":
            sensor_positions.append((col_idx, str(val).strip()))

    n_headers = len(header_row) if header_row else 0

    if not sensor_positions:
        # gecombineerde case: no sensor names in Row 1 → single group spanning all columns
        wn_idx = None
        for i in range(n_headers):
            nh = norm_header(str(header_row[i]) if header_row[i] is not None else "")
            if ("waterniveau" in nh or "grondwaterstand" in nh) and ("mnap" in nh or "nap" in nh):
                wn_idx = i
                break
        return [{"sensor_id": "unknown", "wn_idx": wn_idx, "start_col": 0, "end_col": n_headers}]

    groups = []
    for idx, (start_col, sensor_id) in enumerate(sensor_positions):
        if idx + 1 < len(sensor_positions):
            end_col = sensor_positions[idx + 1][0]
        else:
            end_col = n_headers

        # Find Waterniveau/Grondwaterstand within this group's column range
        wn_idx = None
        for i in range(start_col, end_col):
            if i >= n_headers:
                break
            nh = norm_header(str(header_row[i]) if header_row[i] is not None else "")
            if ("waterniveau" in nh or "grondwaterstand" in nh) and ("mnap" in nh or "nap" in nh):
                wn_idx = i
                break

        groups.append({
            "sensor_id": sensor_id,
            "wn_idx": wn_idx,
            "start_col": start_col,
            "end_col": end_col,
        })

    return groups


def process_workbook(xlsx_path: Path) -> list:
    print(f"Bestand: {xlsx_path.name}")

    origin_stem = xlsx_path.stem
    out_folder = dataset_root / origin_stem / "only_csv"
    out_folder.mkdir(parents=True, exist_ok=True)

    wb = load_workbook(xlsx_path, data_only=True, read_only=True)
    written = 0
    written_paths = set()
    summary_rows = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows_iter = ws.iter_rows(values_only=True)
        try:
            title_row = next(rows_iter)
        except StopIteration:
            continue
        try:
            header_row = next(rows_iter)
        except StopIteration:
            continue

        base_title = first_non_empty(title_row)
        safe_title = sanitize_filename(base_title)

        headers = [str(h) if h is not None else "" for h in header_row]
        norm_headers = [norm_header(h) for h in headers]

        # Find timestamp column
        ts_idx = None
        for i, nh in enumerate(norm_headers):
            if nh in ("timestamp", "time", "datetime"):
                ts_idx = i
                break

        # Detect sensor groups
        groups = detect_sensor_groups(title_row, header_row)
        wl_groups = [g for g in groups if g["wn_idx"] is not None]

        if ts_idx is None or not wl_groups:
            print(f"  Tabblad: {sheet_name} -> skipped (missing Timestamp or Waterniveau/Grondwaterstand column)")
            continue

        # Collect all data rows once (needed for multi-sensor iteration)
        data_rows = list(rows_iter)

        series_name = base_title
        summary_row = {
            "series_name": series_name,
            "nr_sensors": len(groups),
            "nr_series": len(wl_groups),
        }

        for sensor_num, group in enumerate(wl_groups, start=1):
            wn_idx = group["wn_idx"]

            records = []
            for row in data_rows:
                if row is None:
                    continue

                ts_val = row[ts_idx] if ts_idx < len(row) else None
                wn_val = row[wn_idx] if wn_idx < len(row) else None

                ts = to_datetime(ts_val, wb.epoch)
                if ts is None:
                    try:
                        ts = pd.to_datetime(ts_val, dayfirst=True, errors="coerce")
                        if pd.isna(ts):
                            continue
                    except Exception:
                        continue

                wn = pd.to_numeric(wn_val, errors="coerce")
                if pd.isna(wn):
                    continue

                records.append((ts, wn))

            if not records:
                print(f"  Tabblad: {sheet_name} sensor {sensor_num}/{len(wl_groups)} ({group['sensor_id']}) -> no valid rows")
                continue

            df = pd.DataFrame(records, columns=["Time", "head"]).set_index("Time")
            df.sort_index(inplace=True)

            summary_row[f"start_{sensor_num}"] = df.index.min()

            # Filename: no suffix for single-sensor, _sensor_N for multi-sensor
            if len(wl_groups) == 1:
                out_path = out_folder / f"{safe_title}.csv"
            else:
                out_path = out_folder / f"{safe_title}_sensor_{sensor_num}.csv"

            if out_path in written_paths:
                safe_sheet = sanitize_filename(sheet_name)
                out_path = out_folder / f"{out_path.stem} [{safe_sheet}].csv"

            df.to_csv(out_path, index=True, index_label="Time")
            written_paths.add(out_path)
            print(f"  Tabblad: {sheet_name} sensor {sensor_num}/{len(wl_groups)} ({group['sensor_id']}) -> Saved {out_path.name} ({len(df)} rows)")
            written += 1

        summary_rows.append(summary_row)

    wb.close()
    print(f"Done. Saved {written} CSVs to {out_folder}\n")
    return summary_rows


# ---------- Run over folder ----------
if not in_folder.exists():
    raise FileNotFoundError(f"Map niet gevonden: {in_folder}")

xlsx_files = sorted([p for p in in_folder.iterdir() if p.suffix.lower() == ".xlsx"])
if not xlsx_files:
    print("No Excel files found in", in_folder)
else:
    all_summary_rows = []
    for xfile in xlsx_files:
        rows = process_workbook(xfile)
        all_summary_rows.extend(rows)

    if all_summary_rows:
        summary_df = pd.DataFrame(all_summary_rows)
        start_cols = sorted(
            [c for c in summary_df.columns if c.startswith("start_")],
            key=lambda c: int(c.split("_")[1]),
        )
        ordered_cols = ["series_name", "nr_sensors", "nr_series"] + start_cols
        summary_df = summary_df.reindex(columns=ordered_cols)
        summary_path = dataset_root / "series_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"Summary saved to {summary_path} ({len(summary_df)} rows)")

Bestand: Beemster_86349_1_deel_1.xlsx
  Tabblad: 86349-1 MB049PB01 (B_BE0377+32_ sensor 1/1 (MSLV8_o54177) -> Saved 86349-1 MB049PB01 B_BE0377+32_BIKR_GMW_PB1_F-252.csv (16200 rows)
  Tabblad: 86349-1 MB047PB01 (B_BE0328+2_B sensor 1/1 (MSLV8_o54151) -> Saved 86349-1 MB047PB01 B_BE0328+2_BIT_GMW_PB1_F-651.csv (16336 rows)
  Tabblad: 86349-1 MB045PB01 (B_BE0328+3_B sensor 1/1 (MSLV8_o54169) -> Saved 86349-1 MB045PB01 B_BE0328+3_BUKR_GMW_PB1_F-349.csv (16322 rows)
  Tabblad: 86349-1 MB043PB01 (B_BE0263+75_ sensor 1/1 (MSLV8_o54208) -> Saved 86349-1 MB043PB01 B_BE0263+75_BIT_GMW_PB1_F-452.csv (16304 rows)
  Tabblad: 86349-1 MB042PB01 (B_BE0263+75_ sensor 1/1 (MSLV8_o54202) -> Saved 86349-1 MB042PB01 B_BE0263+75_BIKR_GMW_PB1_F-418.csv (16297 rows)
  Tabblad: 86349-1 MB038PB01 (B_BE0254+96_ sensor 1/1 (MSLV8_o54173) -> Saved 86349-1 MB038PB01 B_BE0254+96_BIKR_GMW_PB1_F-396.csv (16326 rows)
  Tabblad: 86349-1 MB035PB01 (B_BE0242+8_B sensor 1/1 (MSLV8_o54162) -> Saved 86349-1 MB035PB01 B_BE02